<a href="https://colab.research.google.com/github/Avi0095/Deep-Learning/blob/main/Autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
from torch.autograd import Variable

In [2]:
movies=pd.read_csv('/content/movies.dat',sep='::',header=None, engine='python',encoding='latin-1')

In [3]:
users=pd.read_csv('/content/users.dat',sep='::',header=None, engine='python',encoding='latin-1')


In [4]:
ratings=pd.read_csv('/content/ratings.dat',sep='::',header=None, engine='python',encoding='latin-1')

In [6]:
training_set=pd.read_csv('/content/u1.base',delimiter='\t')
training_set=np.array(training_set,dtype='int')
test_set=pd.read_csv('/content/u1.test',delimiter='\t')
test_set=np.array(test_set,dtype='int')

In [7]:
nb_users=int(max(max(training_set[:,0]),max(test_set[:,0])))
nb_movies=int(max(max(training_set[:,1]),max(test_set[:,1])))
print(nb_users)
print(nb_movies)

943
1682


In [8]:
import numpy as np
training_set = np.array(training_set)
test_set = np.array(test_set)

def convert(data):
    data = np.array(data)
    new_data = []
    for id_users in range(1, nb_users + 1):
        id_movies = data[:,1][data[:,0] == id_users].astype(int)
        id_ratings = data[:,2][data[:,0] == id_users]
        ratings = np.zeros(nb_movies)
        ratings[id_movies - 1] = id_ratings
        new_data.append(list(ratings))
    return new_data

training_set = convert(training_set)
test_set = convert(test_set)

In [9]:
training_set=torch.FloatTensor(training_set)
test_set=torch.FloatTensor(test_set)

creating the architecture of the neural network

In [12]:
class SAE(nn.Module):
  def __init__(self, ):
    super(SAE,self).__init__()
    self.fc1=nn.Linear(nb_movies,20)
    self.fc2=nn.Linear(20,10)
    self.fc3=nn.Linear(10,20)
    self.fc4=nn.Linear(20,nb_movies)
    self.activation=nn.Sigmoid()
  def forward(self,x):
    x=self.activation(self.fc1(x))
    x=self.activation(self.fc2(x))
    x=self.activation(self.fc3(x))
    x=self.fc4(x)
    return x
sae=SAE()
criterion=nn.MSELoss()
optimizer=optim.RMSprop(sae.parameters(),lr=0.01,weight_decay=0.5)

training the SAE

In [14]:
nb_epoch=200
for epoch in range(1,nb_epoch+1):
  train_loss=0
  s=0.
  for id_user in range(nb_users):
    input=Variable(training_set[id_user]).unsqueeze(0)
    target=input.clone()
    if torch.sum(target.data>0)>0:
      output=sae(input)
      target.require_grad=False
      output[target==0]=0
      loss=criterion(output,target)
      mean_corrector=nb_movies/float(torch.sum(target.data>0)+1e-10)
      loss.backward()
      train_loss +=np.sqrt(loss.item()*mean_corrector)
      s+=1.
      optimizer.step()
  print('epoch:'+str(epoch)+'loss:'+str(train_loss/s))

epoch:1loss:1.7663341441523313
epoch:2loss:1.0964041551955859
epoch:3loss:1.0534558559526248
epoch:4loss:1.038371359867635
epoch:5loss:1.0307602235658868
epoch:6loss:1.0265176890420484
epoch:7loss:1.0237022759629923
epoch:8loss:1.0219800262048595
epoch:9loss:1.0208941045081688
epoch:10loss:1.019670227238294
epoch:11loss:1.0187070363991735
epoch:12loss:1.0183571314626936
epoch:13loss:1.017679097285889
epoch:14loss:1.0173589151327518
epoch:15loss:1.0171172973019846
epoch:16loss:1.0167930061444652
epoch:17loss:1.0168802786464581
epoch:18loss:1.0165050755350853
epoch:19loss:1.0163387313744447
epoch:20loss:1.0159876392074023
epoch:21loss:1.0161435628196713
epoch:22loss:1.0156057206591946
epoch:23loss:1.015955561937754
epoch:24loss:1.0157842599972833
epoch:25loss:1.0155646940451108
epoch:26loss:1.0154629288909558
epoch:27loss:1.0153081198324296
epoch:28loss:1.0148303858817602
epoch:29loss:1.0138588936910045
epoch:30loss:1.01208305687674
epoch:31loss:1.0101639685261135
epoch:32loss:1.00916684

testing the SAE

In [16]:

  test_loss=0
  s=0.
  for id_user in range(nb_users):
    input=Variable(training_set[id_user]).unsqueeze(0)
    target=Variable(test_set[id_user]).unsqueeze(0)
    if torch.sum(target.data>0)>0:
      output=sae(input)
      target.require_grad=False
      output[target==0]=0
      loss=criterion(output,target)
      mean_corrector=nb_movies/float(torch.sum(target.data>0)+1e-10)
      test_loss +=np.sqrt(loss.item()*mean_corrector)
      s+=1.
  print('test loss: '+str(test_loss/s))

test loss: 0.9534375827592566
